# Semenov et al. (2025): a metadata survey of the paper's other dandisets

Paper. Semenov I, Kim V, Silkuniene G, Pakhomov AG (2025), *Cell Reports Physical Science* 6,
102660, [doi:10.1016/j.xcrp.2025.102660](https://doi.org/10.1016/j.xcrp.2025.102660).

The claims on Figures 1, 2, 3B, 5, 6 and S2 have `data: elsewhere`, in dandisets 001032 to 001035,
001271 and 001274. This attempt did not test them (attempt record). This notebook prints the facts
about those dandisets that the record's notes and findings state: their sizes, the image format,
the solution and drug named in each session description, and whether 001034 repeats 001032. It
reads every file of 001030, 001032 to 001035 and 001271 (about 10 minutes), and only the archive's
listing of 001274, which holds 25 GB of strobe images.

| Claim | Tested here |
|---|---|
| none | no; the survey supports the notes of blocked claims |

No random numbers are used.

In [ ]:
import hashlib
import json
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor

import h5py
import pandas as pd
import remfile

pd.set_option("display.width", 200)
API = "https://api.dandiarchive.org/api"
# The most recent published version of each dandiset; the paper lists them without a version.
VERSIONS = {"001030": "0.240520.1924", "001032": "0.240527.1432", "001033": "0.240527.1432",
            "001034": "0.240527.1431", "001035": "0.240527.1432", "001271": "0.250805.1352",
            "001273": "0.250805.1356", "001274": "0.250805.1401"}


def get(url):
    with urllib.request.urlopen(url) as f:
        return json.load(f)


sizes = pd.DataFrame([{"dandiset": d, "version": v, **{k: get(f"{API}/dandisets/{d}/versions/{v}/info/")[k]
                                                       for k in ("asset_count", "size")}}
                      for d, v in VERSIONS.items()])
sizes["size_gb"] = (sizes.pop("size") / 1e9).round(2)
print(sizes.to_string(index=False))

DANDI:001273, the strobe images of Figure 4, holds 10.6 GB; DANDI:001274 holds 25.0 GB. The
fast-imaging dandisets are 0.05 to 0.2 GB each.

In [ ]:
def read(asset):
    """Session description, subject, image format and a hash of the images of one file."""
    with h5py.File(remfile.File(f"{API}/assets/{asset['asset_id']}/download/"), "r") as h:
        s = h["acquisition"][list(h["acquisition"])[0]]
        data = s["data"][:]
        return {"path": asset["path"], "subject_id": h["general/subject/subject_id"][()].decode(),
                "description": h["session_description"][()].decode(), "frames": data.shape[0],
                "pixels": "x".join(map(str, data.shape[1:])), "rate_hz": float(s["starting_time"].attrs["rate"]),
                "images_sha1": hashlib.sha1(data.tobytes()).hexdigest()}


started = time.time()
rows = []
for d in ("001030", "001032", "001033", "001034", "001035", "001271"):
    assets = get(f"{API}/dandisets/{d}/versions/{VERSIONS[d]}/assets/?page_size=1000")["results"]
    with ThreadPoolExecutor(16) as ex:
        rows += [{"dandiset": d, **r} for r in ex.map(read, assets)]
files = pd.DataFrame(rows)
print(f"{len(files)} files read in {time.time() - started:.0f} s")

In [ ]:
files["NaCl"] = files.description.str.contains("140 NaCl")
files["NaGluconate"] = files.description.str.contains("NaGluconate")
files["TTX_present"] = files.description.str.contains("presence of the sodium channel inhibitor 1 µM tetrodotoxin")
files["TTX_absent"] = files.description.str.contains("without tetrodotoxin")
files["states_250_ms"] = files.description.str.contains(r"250 ?ms")
files["states_2_s"] = files.description.str.contains("lasts 2 s")
print(files.groupby("dandiset").agg(files=("path", "size"), subject_ids=("subject_id", "nunique"),
                                    NaCl=("NaCl", "sum"), NaGluconate=("NaGluconate", "sum"),
                                    states_250_ms=("states_250_ms", "sum"), states_2_s=("states_2_s", "sum"),
                                    TTX_present=("TTX_present", "sum"), TTX_absent=("TTX_absent", "sum")).to_string())
print("\nImage format per dandiset:")
print(files.groupby(["dandiset", "pixels", "rate_hz"]).size().rename("files").to_string())

Every file is a stack of 16 x 16 pixel images, at 3,086.4 frames/s in 001032 to 001035 and 001271
and 3,134.8 frames/s in 001030. Every session description of 001032 to 001035 names the standard
140 mM NaCl solution (those of 001032 to 001034 also state a 250-ms stimulus; those of 001035 give
no duration); the only sodium gluconate sessions are the 17 of 001030, all with a 2-s stimulus. The NaGlu thresholds of Figure 3B (claim fig3b-low-conductivity-halves-threshold) do not
appear to be deposited. DANDI:001271 holds 287 sessions of 20 subject ids, 71 recorded in the
presence of tetrodotoxin and 216 without it (vehicle control).

In [ ]:
a = files[files.dandiset == "001032"]
b = files[files.dandiset == "001034"]
pairs = a.merge(b, on="images_sha1", suffixes=("_001032", "_001034"))
print("Files:", len(a), "in 001032,", len(b), "in 001034; distinct image arrays", a.images_sha1.nunique(),
      "and", b.images_sha1.nunique(), "; arrays found in both:", len(pairs))
strip = lambda s: s.str.replace(r"Subject ID: \S+", "", regex=True)
print("Matched pairs with the same session description apart from the subject id:",
      int((strip(pairs.description_001032) == strip(pairs.description_001034)).sum()))
print("Subject ids paired (001032, 001034):")
print(pairs.groupby(["subject_id_001032", "subject_id_001034"]).size().rename("files").to_string())

Every one of the 178 image arrays of 001034 is also in 001032, with the same session description,
under subject ids ending in _3 in place of _1. The two dandisets hold the same recordings.

## Summary

This notebook tests no claim. It prints what the attempt record states about the paper's other
dandisets: DANDI:001273 holds 10.6 GB and 001274 25.0 GB; the fast-imaging files are 16 x 16 pixel
stacks at 3,086.4 frames/s (3,134.8 in 001030); every session of 001032 to 001035 is in NaCl
solution, so the sodium gluconate thresholds of Figure 3B do not appear to be deposited; 001034
repeats the 178 recordings of 001032 under other subject ids; 001271 holds 287 sessions, 71 with
tetrodotoxin.